# Protocolo de validação

Neste notebook é definido o protocolo de validação cruzada utilizado em
todos os experimentos do projeto.

Devido à presença de respostas textuais repetidas no conjunto de dados,
utiliza-se `StratifiedGroupKFold`, agrupando as instâncias pelo texto da
resposta. Dessa forma, ocorrências da mesma resposta permanecem sempre
no mesmo fold, evitando vazamento de dados entre treinamento e validação.

São utilizados 5 folds, com embaralhamento e semente fixa para garantir
reprodutibilidade.

In [58]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.folds import criar_folds

In [59]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train.xlsx"

df = pd.read_excel(DATA_PATH, sheet_name="train")

df["resp_text"] = df["resp_text"].astype(str)

df.shape


(20092, 2)

In [60]:

df["fold"] = criar_folds(df)

df.head()


,resp_text,clarity,fold
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5,3
1,"Prezada cidadã, As informações sobre óbitos ...",c1,3
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1,4
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234,4
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234,0


In [61]:

df["fold"].value_counts().sort_index()

fold
0    4019
1    4019
2    4019
3    4016
4    4019
Name: count, dtype: int64

In [62]:
for fold in sorted(df["fold"].unique()):
    train = df[df["fold"] != fold]
    validation = df[df["fold"] == fold]

    train_texts = set(train["resp_text"])
    validation_texts = set(validation["resp_text"])

    overlap = train_texts.intersection(validation_texts)

    print(
        f"Fold {fold}: "
        f"{len(train)} treino | "
        f"{len(validation)} validação | "
        f"textos compartilhados = {len(overlap)}"
    )

Fold 0: 16073 treino | 4019 validação | textos compartilhados = 0
Fold 1: 16073 treino | 4019 validação | textos compartilhados = 0
Fold 2: 16073 treino | 4019 validação | textos compartilhados = 0
Fold 3: 16076 treino | 4016 validação | textos compartilhados = 0
Fold 4: 16073 treino | 4019 validação | textos compartilhados = 0


In [63]:
distribuicao_folds = pd.crosstab(
    df["fold"],
    df["clarity"],
    normalize="index",
) * 100

distribuicao_folds.round(2)

clarity,c1,c234,c5
fold,,,
0,31.58,34.11,34.31
1,31.58,34.11,34.31
2,31.60,34.11,34.29
3,31.60,34.11,34.29
4,31.60,34.09,34.31


In [64]:
pd.crosstab(
    df["fold"],
    df["clarity"],
)

clarity,c1,c234,c5
fold,,,
0,1269,1371,1379
1,1269,1371,1379
2,1270,1371,1378
3,1269,1370,1377
4,1270,1370,1379


In [65]:
for fold in sorted(df["fold"].unique()):
    train_texts = set(
        df.loc[df["fold"] != fold, "resp_text"]
    )

    validation_texts = set(
        df.loc[df["fold"] == fold, "resp_text"]
    )

    assert train_texts.isdisjoint(validation_texts), (
        f"Há vazamento de textos no fold {fold}."
    )

print("Validação concluída: nenhum texto aparece em treino e validação.")

SPLITS_PATH = PROJECT_ROOT / "data" / "splits" / "folds.csv"

fold_assignments = pd.DataFrame({
    "row_index": df.index,
    "fold": df["fold"],
})

fold_assignments.to_csv(
    SPLITS_PATH,
    index=False,
)

print(f"Folds salvos em: {SPLITS_PATH}")

Validação concluída: nenhum texto aparece em treino e validação.
Folds salvos em: c:\Users\valer\Projects\PLN\Processamento-de-Linguagem-Natural\data\splits\folds.csv


In [66]:
pd.read_csv(SPLITS_PATH).head()

,row_index,fold
0,0,3
1,1,3
2,2,4
3,3,4
4,4,0


In [67]:
for fold in sorted(df["fold"].unique()):
    train = df[df["fold"] != fold]
    validation = df[df["fold"] == fold]

    overlap = set(train["resp_text"]) & set(validation["resp_text"])

    print(
        fold,
        len(train),
        len(validation),
        len(overlap)
    )

0 16073 4019 0
1 16073 4019 0
2 16073 4019 0
3 16076 4016 0
4 16073 4019 0


In [68]:
distribuicao_folds.round(2)

clarity,c1,c234,c5
fold,,,
0,31.58,34.11,34.31
1,31.58,34.11,34.31
2,31.60,34.11,34.29
3,31.60,34.11,34.29
4,31.60,34.09,34.31


In [71]:
from src.evaluation.folds import carregar_folds

df_teste = pd.read_excel(
    PROJECT_ROOT / "data" / "raw" / "train.xlsx",
    sheet_name="train",
)

df_teste["resp_text"] = df_teste["resp_text"].astype(str)

df_teste = carregar_folds(
    df_teste,
    PROJECT_ROOT / "data" / "splits" / "folds.csv",
)

df_teste.head()

,resp_text,clarity,fold
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5,3
1,"Prezada cidadã, As informações sobre óbitos ...",c1,3
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1,4
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234,4
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234,0
